In [1]:
# Imports
import pandas as pd
from random import randint
from src import *
from src.simulator import SIMULATOR
import numpy as np

In [2]:
# --------------------------------------------
#               INIT & CONFIG
# --------------------------------------------
sim = SIMULATOR()

DEBUG = 1
MAX_ITER = 300000

# DISCO-CGRA Parameters
nRCs = 4
nElementsPerVWRSlice = 32
nColsCGRA = 2

In [3]:
# --------------------------------------------
#               KERNEL CONFIGURATION
# --------------------------------------------
kernel_path = './kernels/gemm_disco/32x32-tiling/'
kernel_number = 1 
column_usage = [True, True] 
nInstrPerCol = 45
imem_add_start = 0 
srf_spm_addres = 0 
version="_2col"

# Block size 32x32
BLOCK_SIZE = 32

sim.kernel_config(column_usage, nInstrPerCol, imem_add_start, srf_spm_addres, kernel_number)

In [ ]:
# --------------------------------------------
#               DATA
# --------------------------------------------
data = np.load(kernel_path + "data.npz")

A = data["A"]
B = data["B"]
C = data["C"]
expected_res = data["D"]

ROWS_A = int(data["ROWS_A"])
COLS_A = int(data["COLS_A"])
COLS_B = int(data["COLS_B"])
ALPHA  = int(data["ALPHA"])
BETA   = int(data["BETA"])

B_t = (B.reshape(COLS_A, COLS_B)).T.flatten()

print(A)
print(B)
print(C)

print(B_t)

In [5]:
def printAsMatrix(array, rows, cols):
    for i in range(rows):
        print(array[i * cols:(i + 1) * cols])

In [ ]:
# --------------------------------------------
#              COMPILE ASM TO HEX
# --------------------------------------------
sim.compileAsmToHex(kernel_path, kernel_number, version=version)

# --------------------------------------------
#          LOAD KERNEL INSTRUCTIONS
# --------------------------------------------

# This needs the hex instructions, if you don't provide them, generate then compiling the asm
sim.kernel_load(kernel_path, version=version + "_autogen", kernel_number=kernel_number)

In [7]:
# --------------------------------------------
#          FILL SPM
# --------------------------------------------
# Default SPM lines
srf_spm_line = 0
nLinesPerMatrix = 8

a_spm_line_ini = 1
b_spm_line_ini = a_spm_line_ini + nLinesPerMatrix
c_spm_line_ini = b_spm_line_ini + nLinesPerMatrix

# A
a_spm_line = a_spm_line_ini
a_row = 0
for r in range(nLinesPerMatrix):
    sim.setSPMLine(a_spm_line, A[a_row*COLS_A:(a_row+4)*COLS_A].copy())
    a_spm_line += 1
    a_row+=4

# B
b_spm_line = b_spm_line_ini
b_col = 0
for c in range(nLinesPerMatrix):
    sim.setSPMLine(b_spm_line, B_t[b_col*COLS_A : (b_col+4)*COLS_A].copy())
    b_spm_line += 1
    b_col+=4

#C
c_spm_line = c_spm_line_ini
c_row = 0
for r in range(nLinesPerMatrix):
    sim.setSPMLine(c_spm_line, C[c_row*COLS_B:(c_row+4)*COLS_B].copy())
    c_spm_line += 1
    c_row+=4

In [8]:
# SRF
# --------------------------------------------
# SRF0 = SPMA
# SRF1 = SPMB
# SRF2 = SPMC
# SRF3 = itL1 (row blocking)
# SRF4 = itL2 (col blocking)
# SRF5 = itL3 (inner dimension)
# SRF6 = alpha
# SRF7 = beta
# --------------------------------------------

# Default SRF values
srf = [0 for i in range(N_ELEMS_PER_VWR)]

# Col 0
srf[0]  = a_spm_line_ini 
srf[1]  = b_spm_line_ini 
srf[2]  = c_spm_line_ini
srf[3]  = 3 # Share 8 lines between the 2 cols
srf[4]  = 7 # Go through every line of B
srf[5]  = 31 # Go through every element of B
srf[6]  = ALPHA
srf[7]  = BETA
# Col 1
srf[8]  = a_spm_line_ini + 4 # Compute the next rows of C
srf[9]  = b_spm_line_ini
srf[10] = c_spm_line_ini + 4 # Compute the next rows of C
srf[11] = 3 # Share 8 lines between the 2 cols
srf[12] = 7
srf[13] = 31
srf[14] = ALPHA
srf[15] = BETA


sim.setSPMLine(srf_spm_line, srf.copy())

In [ ]:
# --------------------------------------------
#               SIMULATE EXECUTION
# --------------------------------------------
show_lcu = []
show_srf = []
show_lsu = []
show_rcs = [[],[],[],[]]
show_mxcu = []
display_ops = [show_lcu, show_lsu, show_mxcu, show_rcs, show_srf]

# Launch kernel
sim.run(kernel_number, display_ops=display_ops, max_iter=MAX_ITER)



In [10]:
# Get kernel results
disco_out = []
c_spm_line = c_spm_line_ini
for i in range(nLinesPerMatrix):
    disco_out.extend(c_int32(x).value for x in sim.getSPMLine(c_spm_line))
    c_spm_line+=1


In [ ]:
# Verify results
errors = 0
for i in range(len(expected_res)):
    if expected_res[i] != disco_out[i]:
        errors+=1
    
if errors == 0:
    print("The result is correct!")
else:
    print("Oops, something went wrong. There are " + str(errors) + " errors (out of " + str(len(expected_res)) + " elements).")
    print("DISCO out:")
    printAsMatrix(disco_out, ROWS_A, COLS_B)
    print("Expected result:")
    printAsMatrix(expected_res, ROWS_A, COLS_B)
    print("A:")
    printAsMatrix(A, ROWS_A, COLS_A)
    print("B_t:")
    printAsMatrix(B_t, COLS_B, COLS_A)
    print("C:")
    printAsMatrix(C, ROWS_A, COLS_B)